# CV Intelligent — Version Sans CrewAI

Pipeline IA direct : **3 appels API** au lieu de 12+.


## 1. Installation


In [22]:
!pip install openai pdfplumber python-dotenv pydantic -q


D�finition de macro non valide.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\rayan\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


## 2. Configuration


In [23]:
import os, json, re, time
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List, Optional
import pdfplumber
from openai import OpenAI, RateLimitError

load_dotenv()

client = OpenAI(
    api_key=os.getenv('OPENROUTER_API_KEY'),
    base_url='https://openrouter.ai/api/v1'
)

# Fallback automatique si un modele est rate-limited
MODELES_GRATUITS = [
    "openai/gpt-oss-120b:free",
    "meta-llama/llama-3.3-70b-instruct:free",
    "nousresearch/hermes-3-llama-3.1-405b:free",
    "google/gemma-3-27b-it:free",
    "google/gemma-3-12b-it:free",
    "openai/gpt-oss-20b:free",
]


def appeler_llm(system: str, user: str, max_tokens: int = 2048) -> str:
    """Appel LLM avec fallback automatique sur plusieurs modeles gratuits."""
    for model in MODELES_GRATUITS:
        try:
            print(f"  -> Essai : {model}")
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": user},
                ],
                temperature=0.1,
                max_tokens=max_tokens,
            )
            print(f"  OK avec {model}")
            return response.choices[0].message.content
        except RateLimitError:
            print(f"  Rate-limited sur {model}, essai suivant...")
            time.sleep(2)
        except Exception as e:
            print(f"  Erreur sur {model}: {e}")
            time.sleep(2)
    raise RuntimeError("Tous les modeles sont indisponibles. Reessaie dans 1 minute.")


print("Configuration OK — fallback sur", len(MODELES_GRATUITS), "modeles gratuits.")


Configuration OK — fallback sur 6 modeles gratuits.


## 3. Modeles de Donnees


In [24]:
class Experience(BaseModel):
    titre: Optional[str] = None
    entreprise: Optional[str] = None
    duree: Optional[str] = None
    lieu: Optional[str] = None
    description: Optional[List[str]] = []

class Formation(BaseModel):
    diplome: Optional[str] = None
    etablissement: Optional[str] = None
    annee: Optional[str] = None
    description: Optional[List[str]] = []

class Projet(BaseModel):
    nom: Optional[str] = None
    description: Optional[List[str]] = []
    technologies: Optional[List[str]] = []

class GroupeCompetence(BaseModel):
    categorie: str
    elements: List[str]

class ProfilCandidat(BaseModel):
    prenom: Optional[str] = None
    nom: Optional[str] = None
    email: Optional[str] = None
    telephone: Optional[str] = None
    ville: Optional[str] = None
    linkedin: Optional[str] = None
    github: Optional[str] = None
    portfolio: Optional[str] = None
    resume: Optional[str] = None
    experiences: Optional[List[Experience]] = []
    formations: Optional[List[Formation]] = []
    competences: Optional[List[GroupeCompetence]] = []
    projets: Optional[List[Projet]] = []
    langues: Optional[List[str]] = []

class AnalyseOffre(BaseModel):
    titre_poste: Optional[str] = None
    entreprise: Optional[str] = None
    competences_requises: Optional[List[str]] = []
    competences_souhaitees: Optional[List[str]] = []
    mots_cles_ats: Optional[List[str]] = []
    resume: Optional[str] = None

class ResultatFinal(BaseModel):
    profil_original: Optional[ProfilCandidat] = None
    profil_optimise: Optional[ProfilCandidat] = None
    analyse_offre: Optional[AnalyseOffre] = None
    score_ats: Optional[int] = None
    commentaire_ats: Optional[str] = None
    lettre_motivation: Optional[str] = None

print('Modeles OK.')


Modeles OK.


## 4. Utilitaires


In [25]:
def lire_pdf(chemin: str) -> str:
    texte = ''
    with pdfplumber.open(chemin) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                texte += t + '\n'
    return texte.strip()

def extraire_json(texte: str) -> dict:
    texte = texte.strip()
    m = re.search(r'```(?:json)?\s*({[\s\S]*?})\s*```', texte)
    if m:
        return json.loads(m.group(1))
    m = re.search(r'({[\s\S]*})', texte)
    if m:
        return json.loads(m.group(1))
    return json.loads(texte)

print('Utilitaires OK.')


Utilitaires OK.


## 5. Pipeline IA (3 appels directs)


In [26]:
def generer_cv(chemin_cv_pdf: str, texte_offre: str) -> ResultatFinal:

    # Etape 1 : Lire le PDF
    print("[1/4] Lecture du PDF...")
    texte_cv = lire_pdf(chemin_cv_pdf)
    if not texte_cv:
        raise ValueError("PDF vide ou illisible.")
    # On augmente la limite a 12000 caracteres pour ne pas couper la fin du CV (les derniers projets)
    texte_cv_court = texte_cv[:12000]

    # Etape 2 : Extraire le profil
    print("[2/4] Extraction du profil (appel 1/3)...")
    schema_profil = json.dumps(ProfilCandidat.model_json_schema(), ensure_ascii=False)
    reponse_profil = appeler_llm(
        system=(
            "Tu es un parseur de CV expert. "
            "Retourne UNIQUEMENT un objet JSON valide, sans texte autour.\n"
            "CRITIQUE : Tu dois extraire l'EXHAUSTIVITE absolue du CV. "
            "N'oublie AUCUN projet (si le CV en a 4 ou 5, extrais-les tous), "
            "AUCUNE experience, et AUCUNE competence. "
            "Garde les descriptions longues et detaillees originales, ne les resume pas."
        ),
        user="Schema attendu :\n" + schema_profil + "\n\nCV a analyser :\n" + texte_cv_court,
        max_tokens=4000
    )
    profil_original = ProfilCandidat(**extraire_json(reponse_profil))

    # Etape 3 : Analyser l'offre
    print("[3/4] Analyse de l'offre (appel 2/3)...")
    schema_offre = json.dumps(AnalyseOffre.model_json_schema(), ensure_ascii=False)
    reponse_offre = appeler_llm(
        system="Tu es un analyste RH. Retourne UNIQUEMENT un objet JSON valide, sans texte autour.",
        user="Schema attendu :\n" + schema_offre + "\n\nOffre d'emploi :\n" + texte_offre,
        max_tokens=1024
    )
    analyse_offre = AnalyseOffre(**extraire_json(reponse_offre))

    # Etape 4 : Adapter CV + score ATS + lettre
    print("[4/4] Adaptation CV + score ATS + lettre (appel 3/3)...")
    profil_json = profil_original.model_dump_json(indent=2)
    offre_json  = analyse_offre.model_dump_json(indent=2)

    system_final = (
        "Tu es un expert RH et redacteur de CV. "
        "Retourne UNIQUEMENT un JSON avec exactement ces 4 cles : "
        "cv_adapte (objet ProfilCandidat optimise pour l'offre), "
        "score_ats (entier 0-100), "
        "commentaire_ats (string court), "
        "lettre_motivation (string 200-300 mots).\n"
        "CRITIQUE POUR cv_adapte :\n"
        "- Tu DOIS conserver TOUS les projets et TOUTES les experiences du profil candidat original.\n"
        "- Ne supprime RIEN.\n"
        "- Tu dois garder les descriptions longues et pertinentes (plusieurs points par experience/projet).\n"
        "Aucun texte autour du JSON."
    )
    user_final = (
        "Profil candidat original :\n" + profil_json +
        "\n\nAnalyse de l'offre :\n" + offre_json +
        "\n\nSchema ProfilCandidat pour cv_adapte :\n" + schema_profil
    )
    reponse_finale = appeler_llm(system=system_final, user=user_final, max_tokens=4000)

    data = extraire_json(reponse_finale)
    profil_optimise = ProfilCandidat(**data.get("cv_adapte", profil_original.model_dump()))
    score_ats       = int(data.get("score_ats", 0))
    commentaire_ats = data.get("commentaire_ats", "")
    lettre          = data.get("lettre_motivation", "")

    print("Pipeline termine avec succes !")
    return ResultatFinal(
        profil_original=profil_original,
        profil_optimise=profil_optimise,
        analyse_offre=analyse_offre,
        score_ats=score_ats,
        commentaire_ats=commentaire_ats,
        lettre_motivation=lettre
    )

print("Pipeline pret.")



Pipeline pret.


## 6. Test


In [27]:
CHEMIN_CV = r'../CV_Rayane_Berrada.pdf'

OFFRE = (
    'Titre : Developpeur Full Stack Junior\n'
    'Entreprise : TechCorp\n\n'
    'Nous recherchons un developpeur Full Stack maitrisant React, Python et FastAPI.\n'
    'Experience MongoDB appreciee. Connaissances IA et integration API un plus.\n'
    'Esprit equipe, autonomie, bonne communication.'
)

resultat = generer_cv(CHEMIN_CV, OFFRE)

print('\n' + '=' * 60)
print(f'Score ATS   : {resultat.score_ats}/100')
print(f'Commentaire : {resultat.commentaire_ats}')
print('=' * 60)
print('\nProfil original extrait du CV :')
print(json.dumps(resultat.profil_original.model_dump(), ensure_ascii=False, indent=2))
print('\nProfil optimise pour offre :')
print(json.dumps(resultat.profil_optimise.model_dump(), ensure_ascii=False, indent=2))
print('\nLettre de motivation :')
print(resultat.lettre_motivation)


[1/4] Lecture du PDF...
[2/4] Extraction du profil (appel 1/3)...
  -> Essai : openai/gpt-oss-120b:free
  OK avec openai/gpt-oss-120b:free
[3/4] Analyse de l'offre (appel 2/3)...
  -> Essai : openai/gpt-oss-120b:free
  OK avec openai/gpt-oss-120b:free
[4/4] Adaptation CV + score ATS + lettre (appel 3/3)...
  -> Essai : openai/gpt-oss-120b:free
  OK avec openai/gpt-oss-120b:free
Pipeline termine avec succes !

Score ATS   : 85/100
Commentaire : Bon alignement avec les compétences clés (React, Python, FastAPI) et expérience en API/IA, manque de MongoDB mais points forts en autonomie et communication.

Profil original extrait du CV :
{
  "prenom": "Rayane",
  "nom": "Berrada",
  "email": "rayane06berrada@gmail.com",
  "telephone": "+33 7 43 63 68 28",
  "ville": "Île-de-France",
  "linkedin": "linkedin.com/in/rayane-berrada-734093352",
  "github": "github.com/iitsh",
  "portfolio": "https://rayaneberrada.vercel.app/",
  "resume": null,
  "experiences": [
    {
      "titre": "Stagiaire Dé

## 7. Sauvegarde


In [28]:
import os, json, subprocess, time
from jinja2 import Template
from pathlib import Path
import pdfplumber

def renderiser_cv_html(profil_d):
    job_title = (profil_d.get("experiences") and len(profil_d["experiences"]) > 0 and profil_d["experiences"][0].get("titre")) or None
    template_html = Path("template_cv.html").read_text(encoding="utf-8")
    return Template(template_html).render(
        nom=profil_d.get("nom", ""), prenom=profil_d.get("prenom", ""), email=profil_d.get("email", ""),
        telephone=profil_d.get("telephone", ""), ville=profil_d.get("ville", ""),
        linkedin=profil_d.get("linkedin", ""), github=profil_d.get("github", ""),
        portfolio=profil_d.get("portfolio", ""), resume=profil_d.get("resume", ""),
        experiences=profil_d.get("experiences", []), formations=profil_d.get("formations", []),
        competences=profil_d.get("competences", []), projets=profil_d.get("projets", []),
        langues=profil_d.get("langues", []), job_title=job_title,
    )

def generer_pdfs(html_cv, html_lettre):
    Path("output/cv_optimise.html").write_text(html_cv, encoding="utf-8")
    Path("output/lettre_motivation.html").write_text(html_lettre, encoding="utf-8")

    script_pdf = """
from playwright.sync_api import sync_playwright
from pathlib import Path
def render_pdf(html_path, pdf_path):
    with sync_playwright() as p:
        browser = p.chromium.launch()
        page = browser.new_page()
        page.goto(Path(html_path).resolve().as_uri())
        page.pdf(path=pdf_path, format="A4", print_background=True, margin={"top": "0", "bottom": "0", "left": "0", "right": "0"})
        browser.close()
render_pdf("output/cv_optimise.html", "output/cv_optimise.pdf")
render_pdf("output/lettre_motivation.html", "output/lettre_motivation.pdf")
"""
    Path("generer_pdf.py").write_text(script_pdf, encoding="utf-8")
    res = subprocess.run(["python", "generer_pdf.py"], capture_output=True, text=True)
    try:
        os.remove("generer_pdf.py")
    except:
        pass
    if res.returncode != 0:
        print("Erreur generation PDF:", res.stderr)

# 1. Sauvegardes initiales
os.makedirs("output", exist_ok=True)
with open("output/profil_original.json", "w", encoding="utf-8") as f:
    json.dump(resultat.profil_original.model_dump(), f, ensure_ascii=False, indent=2)
with open("output/score_ats.json", "w", encoding="utf-8") as f:
    json.dump({"score_ats": resultat.score_ats, "commentaire": resultat.commentaire_ats}, f, ensure_ascii=False, indent=2)

profil_courant = resultat.profil_optimise
tentative = 1
max_tentatives = 10

print("Lancement de la boucle de verification (Objectif: 1 Page MAX)...")

while tentative <= max_tentatives:
    print(f"\nTentative {tentative}/{max_tentatives} - Generation du PDF...")
    
    html_cv = renderiser_cv_html(profil_courant.model_dump())
    
    d = profil_courant.model_dump()
    nom_complet = f"{d.get('prenom', '')} {d.get('nom', '')}".strip()
    paragraphes = "".join(f"<p>{p.strip()}</p>" for p in resultat.lettre_motivation.split("\n") if p.strip())
    html_lettre = f"""<!DOCTYPE html><html lang="fr"><head><meta charset="UTF-8"><style>
    body {{ font-family: "Segoe UI", sans-serif; margin: 60px; color: #1a1a1a; }}
    .header {{ margin-bottom: 40px; }}
    .nom {{ font-size: 22px; font-weight: bold; color: #0f344a; }}
    .contact {{ font-size: 12px; color: #555; margin-top: 4px; }}
    h1 {{ font-size: 16px; color: #0f344a; margin: 30px 0 20px 0; text-transform: uppercase; border-bottom: 2px solid #0f344a; padding-bottom: 8px; }}
    p {{ font-size: 13px; line-height: 1.8; margin-bottom: 14px; text-align: justify; }}
    .signature {{ margin-top: 40px; font-weight: bold; }}
    </style></head><body>
    <div class="header"><div class="nom">{nom_complet}</div><div class="contact">{d.get('email','')} | {d.get('telephone','')} | {d.get('ville','')}</div></div>
    <h1>Lettre de Motivation</h1>{paragraphes}<div class="signature">{nom_complet}</div>
    </body></html>"""
    
    generer_pdfs(html_cv, html_lettre)
    
    nb_pages = 0
    try:
        with pdfplumber.open("output/cv_optimise.pdf") as pdf:
            nb_pages = len(pdf.pages)
    except Exception as e:
        print("Erreur lecture PDF:", e)
        break
        
    print(f"-> Le CV genere fait {nb_pages} page(s).")
    
    if nb_pages == 1:
        print("✅ PARFAIT ! Le CV tient sur 1 page.")
        break
        
    if tentative == max_tentatives:
        print("⚠️ Nombre max de tentatives atteint. On garde ce resultat.")
        break
        
    print("⏳ Le CV est trop long. Appel a l'IA pour raccourcir les descriptions de 20%...")
    schema_profil = json.dumps(ProfilCandidat.model_json_schema(), ensure_ascii=False)
    prompt_sys = (
        "Tu es un expert en redaction de CV. Le profil actuel génère un document de " + str(nb_pages) + " pages. "
        "L'objectif est ABSOLUMENT de tenir sur 1 seule page.\n"
        "Retourne UNIQUEMENT le profil mis a jour au format JSON valide.\n"
        "INSTRUCTIONS CRITIQUES :\n"
        "- Raccourcis TRES FORTEMENT (reduis de 30% a 40%) la longueur du texte des descriptions.\n"
        "- Sois concis. Va droit au but.\n"
        "- Ne SUPPRIME AUCUN projet et AUCUNE experience. Garde les tous, mais rends leurs listes de description plus courtes (fusionne des points si besoin).\n"
        "- Conserve les mots cles ATS importants."
    )
    prompt_user = "Profil a raccourcir :\n" + profil_courant.model_dump_json() + "\n\nSchema attendu :\n" + schema_profil
    
    reponse = appeler_llm(system=prompt_sys, user=prompt_user, max_tokens=4000)
    data = extraire_json(reponse)
    profil_courant = ProfilCandidat(**data)
    
    tentative += 1

print("\nTermine ! Les fichiers sont prets dans le dossier output/")



Lancement de la boucle de verification (Objectif: 1 Page MAX)...

Tentative 1/10 - Generation du PDF...
-> Le CV genere fait 2 page(s).
⏳ Le CV est trop long. Appel a l'IA pour raccourcir les descriptions de 20%...
  -> Essai : openai/gpt-oss-120b:free
  OK avec openai/gpt-oss-120b:free

Tentative 2/10 - Generation du PDF...
-> Le CV genere fait 2 page(s).
⏳ Le CV est trop long. Appel a l'IA pour raccourcir les descriptions de 20%...
  -> Essai : openai/gpt-oss-120b:free
  OK avec openai/gpt-oss-120b:free

Tentative 3/10 - Generation du PDF...
-> Le CV genere fait 1 page(s).
✅ PARFAIT ! Le CV tient sur 1 page.

Termine ! Les fichiers sont prets dans le dossier output/
